# Cell-of-origin correlation analysis: healthy vs cancer

For the `healthy` and `cancer` samples in the pipeline's `results/` folder:

1. loads the per-gene FFT WPS matrix (`results/<sample>/proj/fft_summaries/fft_<sample>_WPS.tsv.gz`, genes x periods 120-280 bp)
2. scores each gene as the mean FFT intensity over 193-199 bp (same as `collapse_fft_wps.py`)
3. correlates the scores with the Tabula Sapiens reference (**baseline = Pearson over all shared genes**, as in `correlate.py`; `pearson_var3k` shown as one alternative)
4. plots correlation-vs-period curves ("S curves") per sample and healthy vs cancer

**Colour code:** red = blood/immune groups (Tabula Sapiens immune compartment), grey = all other groups. The four
negative controls each get their own colour and marker: blue diamonds = erythrocytes, pink squares = platelets,
green triangles = cells behind the blood-retina barrier, yellow triangles = post-mitotic / low-turnover cells
(cardiac and skeletal muscle). The compartment figures at the end use the four Tabula Sapiens compartments instead.

**Four negative controls**, listed group by group in `output/control_groups.txt`:

1. **Erythrocytes** and 2. **platelets** -- anucleate, so no DNA to release. Erythrocytes are kept from all 8 tissues:
   tissue-sampled RBCs carry ambient RNA from the tissue they were dissociated from (`erythrocyte_Liver`'s top genes
   include ALB and FGB), which makes "erythrocytes rank last in *every* tissue" a stronger observation than one blood
   group alone.
3. **Behind the blood-retina barrier** -- retinal neurons and pigment epithelium.
4. **Low turnover** -- cardiac and skeletal muscle, essentially post-mitotic.

Controls 3 and 4 have nuclei but should contribute little cfDNA, and unlike 1 and 2 they are not confounded by a
dividing precursor.

**Erythroid progenitors count as blood/immune, not as control:** they are nucleated, and methylation studies attribute
a large share of healthy plasma cfDNA to the erythroid lineage, so scoring them as a control would penalise a correct
result. Control 1 is soft for the same reason -- platelet RNA comes from megakaryocytes and erythrocyte RNA resembles
erythroblasts, and both precursors do shed cfDNA -- which is why control 2 is shown separately.

## Settings

In [ ]:
from pathlib import Path

# the pipeline's workflow/ directory; ".." works when this notebook sits in workflow/result_analysis/
WORKFLOW_DIR = Path("..")

REF_CSV = WORKFLOW_DIR / "resources/avg_expr_by_celltype_tissue.csv"
# Tabula Sapiens compartment per group (immune / epithelial / endothelial / stromal)
COMPARTMENT_CSV = WORKFLOW_DIR / "resources/celltype_compartment.csv"
RESULTS_DIR = WORKFLOW_DIR / "results"
SAMPLES = ["healthy", "cancer"]

# where plots and the two summary tables are written
OUT_DIR = Path("output")

BAND = (193, 199)                               # period band (bp) used for the gene score
VAR_GENE_COUNTS = [3000]      # one alternative method: Pearson on the 3000 most variable genes
CURVE_METHODS = ["baseline"]  # S curves follow the baseline only
TOP_N = 20
RANK_METHODS = ["baseline"]   # methods used for the report ranking figures, e.g. ["baseline", "pearson_var5k"]
RANK_TOP = 20                 # bars per top / bottom ranking panel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.width", 250)

def fft_path(sample):
    d = RESULTS_DIR / sample / "proj" / "fft_summaries"
    for name in (f"fft_{sample}_WPS.tsv.gz", f"fft_{sample}_WPS.tsv"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"no FFT matrix for '{sample}' in {d}")

assert REF_CSV.exists(), f"reference not found: {REF_CSV}"
FFT_FILES = {s: fft_path(s) for s in SAMPLES}
for s, p in FFT_FILES.items():
    print(f"{s}: {p}")
print("output:", OUT_DIR.resolve())

# one colour per category and per sample, used by every plot below
# four negative-control classes, each with its own colour and marker
ERY = "erythrocyte (anucleate)"
PLT = "platelet (anucleate)"
BARRIER = "behind blood barrier"
LOWTURN = "low turnover"
CONTROL_CATS = [ERY, PLT, BARRIER, LOWTURN]
CAT_COLORS = {"blood/immune": "tab:red", "other": "tab:gray",
              ERY: "#2a78d6", PLT: "#e87ba4", BARRIER: "#1baf7a", LOWTURN: "#eda100"}
CAT_MARKERS = {ERY: "D", PLT: "s", BARRIER: "^", LOWTURN: "v"}      # white-edged, drawn on top
CAT_ZORDER = {ERY: 6, PLT: 6, BARRIER: 5, LOWTURN: 5}
CONTROL_MARK = {ERY: " *", PLT: " +", BARRIER: " ^", LOWTURN: " v"}  # flags in the printed tables
SAMPLE_COLORS = {"healthy": "tab:blue", "cancer": "tab:orange"}

## Load data

In [ ]:
ref = pd.read_csv(REF_CSV, index_col=0)
gene_names = ref.pop("gene_name")
print(f"reference: {ref.shape[0]} genes x {ref.shape[1]} groups")

comp = pd.read_csv(COMPARTMENT_CSV, index_col=0)["compartment"].reindex(ref.columns)
assert comp.notna().all(), f"groups without compartment: {list(comp.index[comp.isna()])}"

# anucleate controls: no nucleus, so no DNA to shed. All tissues are kept -- tissue-sampled RBCs carry ambient RNA
# from the tissue they came from, which makes them a harder test than blood-sampled ones.
# Erythroid progenitors and megakaryocytes are nucleated and DO shed cfDNA: they stay in blood/immune.
CONTROL_TYPES = {
    ERY:     {"erythrocyte"},
    PLT:     {"platelet"},
    BARRIER: {"eye photoreceptor cell", "retinal bipolar neuron", "retina horizontal cell",
              "retinal ganglion cell", "retinal pigment epithelial cell"},       # blood-retina barrier
    LOWTURN: {"cardiac muscle cell", "slow muscle cell", "fast muscle cell",
              "cell of skeletal muscle"},                                        # post-mitotic / long-lived
}
TYPE_TO_CAT = {t: k for k, types in CONTROL_TYPES.items() for t in types}

def category(group):
    cell_type = group.split("_", 1)[0]
    if cell_type in TYPE_TO_CAT:
        return TYPE_TO_CAT[cell_type]
    return "blood/immune" if comp[group] == "immune" else "other"

cat = pd.Series({g: category(g) for g in ref.columns})
cat_color = cat.map(CAT_COLORS)

# what the reference contains
celltypes_all = sorted({g.split("_", 1)[0] for g in ref.columns})
tissues_all = sorted({g.split("_", 1)[1] for g in ref.columns})
print(f"reference: {ref.shape[0]} genes x {len(ref.columns)} (cell type, tissue) groups")
print(f"           {len(celltypes_all)} distinct cell types, {len(tissues_all)} tissues "
      f"(not every combination exists)\n")
print(cat.value_counts().to_string(), "\n")
print(comp.value_counts().to_string())

# exact membership of every control class -> text file + table
lines = [f"Reference: {len(ref.columns)} groups, {len(celltypes_all)} cell types, {len(tissues_all)} tissues", ""]
for k in CONTROL_CATS:
    members = sorted(cat.index[cat == k])
    types = sorted({g.split("_", 1)[0] for g in members})
    lines += [f"{k}  ({len(members)} groups, {len(types)} cell types)",
              f"  cell types: {', '.join(types)}",
              f"  groups:     {', '.join(members)}", ""]
(OUT_DIR / "control_groups.txt").write_text("\n".join(lines))
print("\nwrote", OUT_DIR / "control_groups.txt")
display(pd.DataFrame([{"control": k, "groups": int((cat == k).sum()),
                       "cell types": ", ".join(sorted(CONTROL_TYPES[k])),
                       "tissues": ", ".join(sorted({g.split("_", 1)[1] for g in cat.index[cat == k]}))}
                      for k in CONTROL_CATS]))

def load_fft(path):
    fft = pd.read_csv(path, sep="\t", index_col=0)
    fft.columns = fft.columns.astype(float)
    return fft[sorted(fft.columns)]

fft = {s: load_fft(p) for s, p in FFT_FILES.items()}
scores = {}
for s, m in fft.items():
    band_cols = [c for c in m.columns if BAND[0] <= c <= BAND[1]]
    scores[s] = m[band_cols].mean(axis=1)
    print(f"{s}: {m.shape[0]} genes x {m.shape[1]} periods, band {band_cols}, "
          f"{len(ref.index.intersection(m.index))} genes shared with reference, median score {scores[s].median():.1f}")

## Methods

**Every figure in the report uses `baseline`**: Pearson over all shared genes -- exactly what the pipeline's
`correlate.py` computes, and the method described by Snyder et al. 2016 and Stanley et al. 2024.

`pearson_var3k` is carried along as **one alternative**: Pearson restricted to the 3000 genes whose expression varies
most across groups. Dropping genes that are flat across cell types raises the correlations, but it deviates from the
published method, so it appears only in the method-comparison panels, never in the ranking figures.

In [ ]:
def variable_genes(expr, n):
    return expr.var(axis=1).nlargest(n).index

def label(n):
    return f"pearson_var{n // 1000}k" if n >= 1000 and n % 1000 == 0 else f"pearson_var{n}"

def gene_set(method, expr):  # genes a method correlates over
    if method == "baseline":
        return expr.index
    n = int(method.split("var")[1].replace("k", "000"))
    return variable_genes(expr, n)

METHODS = ["baseline"] + [label(n) for n in VAR_GENE_COUNTS]

def correlate(method, expr, score):
    g = gene_set(method, expr)
    e, s = expr.loc[g], score.loc[g]
    if method.startswith("spearman"):
        return e.rank().corrwith(s.rank())
    return e.corrwith(s)

results = {}  # (method, sample) -> correlation per group
for s in SAMPLES:
    shared = ref.index.intersection(scores[s].index)
    for m in METHODS:
        results[(m, s)] = correlate(m, ref.loc[shared], scores[s].loc[shared]).sort_values()
print("methods:", METHODS)

## Method summary

In [ ]:
CTRL_IDX = {k: cat.index[cat == k] for k in CONTROL_CATS}   # groups per control class
rows = []
for m in METHODS:
    for s in SAMPLES:
        r = results[(m, s)]
        rk = r.rank()
        rows.append({"method": m, "sample": s, "min_r": r.min(), "spread": r.max() - r.min(),
                     f"immune_top{TOP_N}": (cat[r.index[:TOP_N]] == "blood/immune").mean(),
                     **{f"{k}_median_rank": rk[idx].median() for k, idx in CTRL_IDX.items()},
                     "erythrocyte_Blood_rank": int(rk["erythrocyte_Blood"]), "top1": r.index[0]})
    rows[-1]["healthy_vs_cancer_rho"] = rows[-2]["healthy_vs_cancer_rho"] = \
        results[(m, "healthy")].corr(results[(m, "cancer")], method="spearman")
summary = pd.DataFrame(rows).set_index(["method", "sample"])
summary.to_csv(OUT_DIR / "method_summary.csv")
summary.round(3)

In [ ]:
# method comparison at a glance
n_groups = ref.shape[1]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
x = np.arange(len(METHODS)); w = 0.38
for i, s in enumerate(SAMPLES):
    sm = summary.xs(s, level="sample").loc[METHODS]
    off = (i - 0.5) * w
    axes[0].bar(x + off, sm["min_r"], w, color=SAMPLE_COLORS[s], label=s)
    axes[1].bar(x + off, sm["spread"], w, color=SAMPLE_COLORS[s])
    axes[2].bar(x + off, sm[f"immune_top{TOP_N}"], w, color=SAMPLE_COLORS[s])
    axes[3].bar(x + off, sm[f"{ERY}_median_rank"], w, color=SAMPLE_COLORS[s])
rho = summary.xs("healthy", level="sample").loc[METHODS, "healthy_vs_cancer_rho"]
axes[1].plot(x, rho * summary["spread"].max(), "k.--", label="healthy vs cancer ρ (scaled)")
axes[3].axhline(n_groups, color="k", ls=":", lw=1); axes[3].axhline(n_groups / 2, color="k", ls=":", lw=0.6)
titles = ["strongest correlation (min r) - lower is stronger", "spread (max − min r) - higher separates more",
          f"fraction of top {TOP_N} that is blood/immune", f"erythrocyte median rank - higher is better (max {n_groups})"]
for ax, t in zip(axes, titles):
    ax.set_xticks(x); ax.set_xticklabels(METHODS, rotation=40, ha="right"); ax.set_title(t, fontsize=10)
axes[0].legend(); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "method_comparison.png", dpi=150)
plt.show()

## Healthy vs cancer, per method

Each dot is a cell-type group: x = correlation in healthy, y = correlation in cancer; dashed line = equal in both.
A tight diagonal cloud = consistent method; the whole cloud off the diagonal = global (likely technical, e.g. depth)
shift; single groups far off the line = candidate sample-specific signal (needs replicates to be meaningful).

In [ ]:
cat_legend = [Line2D([], [], marker=CAT_MARKERS.get(k, "o"), ls="", color=c, label=k, markeredgecolor="white")
              for k, c in CAT_COLORS.items()]
draw_order = {"other": 0, "blood/immune": 1, **{k: 2 + i for i, k in enumerate(CONTROL_CATS)}}
order = cat.sort_values(key=lambda c: c.map(draw_order)).index      # controls drawn last, on top

ncol = 4
nrow = int(np.ceil(len(METHODS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.8 * nrow))
for ax, m in zip(axes.flat, METHODS):
    h, c = results[(m, "healthy")][order], results[(m, "cancer")][order]
    for k in sorted(CAT_COLORS, key=lambda k: draw_order[k]):   # controls bigger, own marker
        sel = cat[order] == k
        ax.scatter(h[sel.values], c[sel.values], s=55 if k in CAT_MARKERS else 10, marker=CAT_MARKERS.get(k, "o"),
                   c=CAT_COLORS[k], alpha=0.9 if k in CAT_MARKERS else 0.6,
                   edgecolors="white" if k in CAT_MARKERS else "none", linewidths=0.6,
                   zorder=CAT_ZORDER.get(k, 2))
    lo, hi = min(h.min(), c.min()), max(h.max(), c.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_title(f"{m}  (ρ = {h.corr(c, method='spearman'):.2f})")
    ax.set_xlabel("healthy r"); ax.set_ylabel("cancer r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=cat_legend, fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scatter_healthy_vs_cancer.png", dpi=150)
plt.show()

## Correlations by cell-type category

Distribution of every group's correlation, split by category, for each method and sample. A useful method separates
blood/immune (more negative) from other groups, and keeps both control sets at or above the "other" level.

In [ ]:
cats = list(CAT_COLORS)
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), sharey=False)
for ax, m in zip(axes.flat, METHODS):
    for i, k in enumerate(cats):
        for j, s in enumerate(SAMPLES):
            vals = results[(m, s)][cat[results[(m, s)].index] == k]
            pos = i * 3 + j
            bp = ax.boxplot(vals, positions=[pos], widths=0.8, patch_artist=True, showfliers=False)
            bp["boxes"][0].set(facecolor=SAMPLE_COLORS[s], alpha=0.5, edgecolor=CAT_COLORS[k])
            ax.scatter(np.random.default_rng(0).normal(pos, 0.08, len(vals)), vals, s=5, color=CAT_COLORS[k], alpha=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks([i * 3 + 0.5 for i in range(len(cats))]); ax.set_xticklabels(cats)
    ax.set_title(m); ax.set_ylabel("r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=[Patch(color=c, alpha=0.5, label=s) for s, c in SAMPLE_COLORS.items()], fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "category_distributions.png", dpi=150)
plt.show()

In [ ]:
# top groups per method as a table; control flags: * erythrocyte, + platelet, ^ barrier, v low turnover
for s in SAMPLES:
    top = pd.DataFrame({m: [g + CONTROL_MARK.get(cat[g], "") for g in results[(m, s)].index[:TOP_N]]
                        for m in METHODS})
    top.index = range(1, TOP_N + 1)
    print(f"=== {s}")
    display(top)

In [ ]:
allcorr = pd.DataFrame({f"{m}__{s}": r for (m, s), r in results.items()})
allcorr.insert(0, "category", cat)
allcorr.index.name = "cell_type"
allcorr.to_csv(OUT_DIR / "correlations_all_methods.csv")
print(f"wrote {OUT_DIR / 'correlations_all_methods.csv'}")

## Report figures: cell-type rankings

Rankings for the methods in `RANK_METHODS`. Groups are named `<cell type>_<tissue>`; the **cell-type level** figures
average each cell type's correlation over the tissues it was measured in (158 cell types instead of 447 groups),
which is easier to read. Colours as everywhere else; blue = healthy, orange = cancer.

With one healthy and one cancer sample, differences between the two are exploratory only.

In [ ]:
def split_group(g):
    cell_type, tissue = g.split("_", 1)
    return cell_type, tissue.replace("_", " ")

def by_celltype(r):
    # mean correlation per cell type over its tissues, most negative first
    ct = pd.Series([split_group(g)[0] for g in r.index], index=r.index)
    agg = r.groupby(ct).agg(["mean", "count"]).sort_values("mean")
    agg["category"] = [category(g) for g in r.groupby(ct).apply(lambda x: x.index[0]).loc[agg.index]]
    return agg

def bar_ranking(ax, values, labels, cats, title):
    y = np.arange(len(values))[::-1]
    ax.barh(y, values, color=[CAT_COLORS[c] for c in cats], alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels([f"{i + 1}. {l}" for i, l in enumerate(labels)], fontsize=8)
    for yi, v in zip(y, values):
        ax.text(v, yi, f"{v:.3f} ", va="center", ha="left" if v < 0 else "right", fontsize=7,
                color="white" if v < 0 else "black")
    ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("correlation (r)"); ax.set_title(title, fontsize=11)

cat_handles = [Patch(color=c, label=k) for k, c in CAT_COLORS.items()]
celltype_rank = {(m, s): by_celltype(results[(m, s)]) for m in RANK_METHODS for s in SAMPLES}

In [ ]:
# top cell types (averaged over tissues), one panel per sample
for m in RANK_METHODS:
    fig, axes = plt.subplots(1, len(SAMPLES), figsize=(7 * len(SAMPLES), 0.3 * RANK_TOP + 1.5))
    for ax, s in zip(axes, SAMPLES):
        top = celltype_rank[(m, s)].iloc[:RANK_TOP]
        labels = [f"{c}  [n={n}]" for c, n in zip(top.index, top["count"])]
        bar_ranking(ax, top["mean"].values, labels, top["category"], f"{s}: top {RANK_TOP} cell types")
    axes[-1].legend(handles=cat_handles, fontsize=8, loc="lower left")
    fig.suptitle(f"Cell-type ranking, {m} (mean r over tissues; n = number of tissues)", fontsize=12)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"ranking_celltypes_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# biggest rank changes between healthy and cancer (cell-type level)
N_SHIFT = 15
for m in RANK_METHODS:
    rh = celltype_rank[(m, "healthy")]["mean"].rank()
    rc = celltype_rank[(m, "cancer")]["mean"].rank()
    shift = (rh - rc.loc[rh.index]).sort_values()   # positive = ranks higher (stronger) in cancer
    sel = pd.concat([shift.head(N_SHIFT), shift.tail(N_SHIFT)])
    fig, ax = plt.subplots(figsize=(8, 0.28 * len(sel) + 1.5))
    y = np.arange(len(sel))
    ax.barh(y, sel.values, color=[CAT_COLORS[celltype_rank[(m, "healthy")].loc[ct, "category"]] for ct in sel.index], alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels([f"{ct}  (#{int(rh[ct])} → #{int(rc[ct])})" for ct in sel.index], fontsize=8)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(f"rank change (positive = stronger in cancer), out of {len(rh)} cell types")
    ax.set_title(f"Largest rank changes healthy → cancer ({m})\nexploratory: one sample per condition", fontsize=11)
    ax.legend(handles=cat_handles, fontsize=8, loc="lower right")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"rank_shift_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# top 10 cell types per sample and method, as a table
tbl = {}
for m in RANK_METHODS:
    for s in SAMPLES:
        top = celltype_rank[(m, s)].iloc[:10]
        tbl[(m, s)] = [f"{c} ({v:.3f})" for c, v in zip(top.index, top["mean"])]
top10 = pd.DataFrame(tbl, index=range(1, 11))
top10.to_csv(OUT_DIR / "top10_celltypes.csv")
top10

## Report figures: rank distributions (simplified versions of Stanley et al. 2024, Fig. 2)

Each group's correlation is turned into a **rank within the sample** (1 = most negative = strongest signal,
447 = weakest). The paper's boxplots are over many samples; with one sample per condition, the boxes here are
over the **groups inside a category** (all cell types of a compartment, all cell types in a tissue, all tissues of a
cell type), shown side by side for healthy (blue) and cancer (orange). Y axes are flipped so better ranks are higher.
Blue diamonds = anucleate groups (erythrocytes in every tissue, platelets); green triangles = barrier /
low-turnover cells (retina, cardiac and skeletal muscle). Both should sit near the weak end.

In [ ]:
REPORT_METHOD = RANK_METHODS[0]
COMP_COLORS = {"immune": "tab:red", "epithelial": "tab:green", "endothelial": "tab:purple", "stromal": "tab:brown"}
ranks = {s: results[(REPORT_METHOD, s)].rank() for s in SAMPLES}   # 1 = most negative
n_groups = len(ranks[SAMPLES[0]])
tissue = pd.Series({g: split_group(g)[1] for g in ref.columns})
celltype = pd.Series({g: split_group(g)[0] for g in ref.columns})

def paired_boxes(ax, keys, members, show_ery=True, vertical=True, legend_outside=True):
    # members(key) -> list of groups; draws a healthy and a cancer box per key
    rng = np.random.default_rng(0)
    for i, k in enumerate(keys):
        groups = members(k)
        for j, s in enumerate(SAMPLES):
            pos = i + (j - 0.5) * 0.38
            vals = ranks[s][groups]
            bp = ax.boxplot(vals, positions=[pos], widths=0.34, patch_artist=True, showfliers=False, vert=vertical)
            bp["boxes"][0].set(facecolor=SAMPLE_COLORS[s], alpha=0.45)
            jit = rng.normal(pos, 0.05, len(vals))
            cats = np.array([cat[g] for g in groups])
            is_ctrl = np.isin(cats, list(CAT_MARKERS))
            xy = (jit, vals) if vertical else (vals, jit)
            ax.scatter(*(np.asarray(a)[~is_ctrl] for a in xy), s=6, color="k", alpha=0.3, zorder=3)
            if show_ery:
                for k, mk in CAT_MARKERS.items():          # controls: own colour, marker and white edge
                    sel = cats == k
                    if sel.any():
                        ax.scatter(*(np.asarray(a)[sel] for a in xy), s=70, marker=mk, color=CAT_COLORS[k],
                                   edgecolors="white", linewidths=0.8, zorder=CAT_ZORDER[k])
    sample_handles = [Patch(color=c, alpha=0.45, label=s) for s, c in SAMPLE_COLORS.items()]
    ctrl_handles = [Line2D([], [], marker=mk, ls="", color=CAT_COLORS[k], markeredgecolor="white", label=k)
                    for k, mk in CAT_MARKERS.items()]
    handles = sample_handles + ctrl_handles
    if legend_outside:
        ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0,
                  frameon=False)
    else:
        ax.legend(handles=handles, fontsize=8)

In [ ]:
# D. ranks by compartment
keys = (comp.groupby(comp).apply(lambda t: ranks["healthy"][t.index].median())
        .sort_values().index.tolist())   # strongest (lowest median rank) first, from the healthy sample
fig, ax = plt.subplots(figsize=(8, 5))
paired_boxes(ax, keys, lambda k: list(comp.index[comp == k]))
ax.set_xticks(range(len(keys))); ax.set_xticklabels([f"{k}\n(n={int((comp == k).sum())})" for k in keys])
for lbl, k in zip(ax.get_xticklabels(), keys):
    lbl.set_color(COMP_COLORS[k])
ax.invert_yaxis(); ax.set_ylabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Correlation rank by compartment ({REPORT_METHOD})")
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_by_compartment_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# E. ranks by tissue, sorted by healthy median rank
keys = tissue.groupby(tissue).apply(lambda t: ranks["healthy"][t.index].median()).sort_values().index
fig, ax = plt.subplots(figsize=(16, 5.5))
paired_boxes(ax, keys, lambda k: list(tissue.index[tissue == k]))
ax.set_xticks(range(len(keys))); ax.set_xticklabels([f"{k} ({int((tissue == k).sum())})" for k in keys], rotation=45, ha="right")
ax.invert_yaxis(); ax.set_ylabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Correlation rank by tissue, all cell types in the tissue ({REPORT_METHOD}); (n) = cell types")
plt.tight_layout(rect=(0, 0, 0.88, 1))    # leave room for the legend outside the axes
fig.savefig(OUT_DIR / f"ranks_by_tissue_{REPORT_METHOD}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# F. immune compartment, ranks by cell type (points = tissues), sorted by healthy median rank
imm = comp.index[comp == "immune"]
keys = celltype[imm].groupby(celltype[imm]).apply(lambda t: ranks["healthy"][t.index].median()).sort_values().index
fig, ax = plt.subplots(figsize=(8, 0.28 * len(keys) + 1.5))
paired_boxes(ax, keys, lambda k: list(imm[celltype[imm] == k]), vertical=False)
ax.set_yticks(range(len(keys))); ax.set_yticklabels([f"{k} ({int((celltype[imm] == k).sum())})" for k in keys], fontsize=7)
ax.invert_yaxis()   # strongest cell types at the top
ax.set_xlabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Immune cell types, rank across tissues\n({REPORT_METHOD}; (n) = tissues)", fontsize=11)
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_immune_celltypes_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# H. macrophage rank by tissue of residence (one value per tissue and sample)
mac = celltype.index[celltype == "macrophage"]
order = ranks["healthy"][mac].sort_values().index
fig, ax = plt.subplots(figsize=(7, 0.32 * len(order) + 1.5))
y = np.arange(len(order))
for s in SAMPLES:
    ax.scatter(ranks[s][order], y, color=SAMPLE_COLORS[s], s=45, label=s, zorder=3)
ax.hlines(y, ranks["healthy"][order], ranks["cancer"][order], color="lightgray", lw=2)
ax.set_yticks(y); ax.set_yticklabels([tissue[g] for g in order])
ax.invert_yaxis()
ax.set_xlim(0, n_groups); ax.set_xlabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Macrophage rank by tissue of residence ({REPORT_METHOD})"); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_macrophage_by_tissue_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# top and bottom of the ranking (cell type + tissue), coloured by compartment, anucleate controls in black
for m in RANK_METHODS:
    fig, axes = plt.subplots(2, len(SAMPLES), figsize=(7.5 * len(SAMPLES), 0.3 * 2 * RANK_TOP + 2.5))
    for j, s in enumerate(SAMPLES):
        r = results[(m, s)]
        for i, (part, sel) in enumerate((("top", r.iloc[:RANK_TOP]), ("bottom", r.iloc[-RANK_TOP:]))):
            ax = axes[i, j]
            start = 1 if part == "top" else len(r) - RANK_TOP + 1
            y = np.arange(len(sel))[::-1]
            colors = [CAT_COLORS[cat[g]] if cat[g] in CAT_MARKERS else COMP_COLORS[comp[g]] for g in sel.index]
            ax.barh(y, sel.values, color=colors, alpha=0.85)
            ax.set_yticks(y)
            ax.set_yticklabels([f"{start + k}. {ct} ({t})" for k, (ct, t) in enumerate(map(split_group, sel.index))], fontsize=7)
            ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("correlation (r)")
            ax.xaxis.set_major_locator(plt.MaxNLocator(5))
            ax.set_title(f"{s}: {part} {RANK_TOP} of {len(r)}", fontsize=10)
    handles = ([Patch(color=c, label=k) for k, c in COMP_COLORS.items()]
               + [Patch(color=CAT_COLORS[k], label=k) for k in CAT_MARKERS])
    fig.suptitle(f"Strongest and weakest cell type / tissue groups ({m})", fontsize=12)
    fig.legend(handles=handles, fontsize=8, loc="upper right", ncol=len(handles))
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(OUT_DIR / f"ranking_top_bottom_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

## S curves (baseline)

The same figure Snyder et al. produce in `cfDNA/expression/plots.R`: for **every period column (120-280 bp)**,
correlate each gene's FFT intensity at that period with each group's expression, then plot correlation against
period. In `plots.R` every tissue is a thin grey line with one tissue highlighted in black; here the thin lines are
the 447 groups and the bold lines are category medians.

The expected shape is an S: positive correlation at short periods, crossing zero, and a **negative dip at the
nucleosome repeat length (193-199 bp, gold band)** for the cell types that shed the cfDNA. The controls should stay
flat -- if they dip as deeply as blood/immune, the dip is not cell-type specific.

In [ ]:
CURVE_METHOD = CURVE_METHODS[0]        # baseline

def period_curves(method, m):
    "correlation of every gene's intensity at each period with every group's expression -> periods x groups"
    shared = ref.index.intersection(m.index)
    g = gene_set(method, ref.loc[shared])
    expr = ref.loc[g]
    return pd.DataFrame({p: expr.corrwith(m.loc[g, p]) for p in m.columns}).T

curves = {s: period_curves(CURVE_METHOD, fft[s]) for s in SAMPLES}
periods = curves[SAMPLES[0]].index.values

def cat_median(curve, k):
    return curve.loc[:, cat[curve.columns] == k].median(axis=1)

# where the blood/immune median dips deepest, per sample
for s in SAMPLES:
    med = cat_median(curves[s], "blood/immune")
    print(f"{s}: blood/immune median r is most negative at {med.idxmin():.0f} bp (r = {med.min():+.3f}); "
          f"at the {BAND[0]}-{BAND[1]} bp band r = {med[(med.index >= BAND[0]) & (med.index <= BAND[1])].mean():+.3f}")

In [ ]:
# one panel per sample: every group as a faint line (plots.R style), category medians drawn on top
fig, axes = plt.subplots(1, len(SAMPLES), figsize=(7.5 * len(SAMPLES), 5), sharey=True, squeeze=False)
for ax, s in zip(axes[0], SAMPLES):
    cur = curves[s]
    ax.axvspan(*BAND, color="gold", alpha=0.3, zorder=0)
    ax.axhline(0, color="k", lw=0.6)
    ax.plot(periods, cur.values, color="0.8", lw=0.4, alpha=0.5, zorder=1)      # all 447 groups
    for k in ["blood/immune"] + CONTROL_CATS:
        ax.plot(periods, cat_median(cur, k), color=CAT_COLORS[k], lw=2.4, zorder=3,
                label=f"{k} (n={(cat == k).sum()})")
    ax.plot(periods, cat_median(cur, "other"), color="dimgray", lw=1.6, ls=":", zorder=2,
            label=f"other (n={(cat == 'other').sum()})")
    ax.set_title(f"{s}", fontsize=12)
    ax.set_xlabel("period (bp)")
axes[0, 0].set_ylabel("correlation with expression (r)")
axes[0, -1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
fig.suptitle(f"Correlation vs period, {CURVE_METHOD}: thin grey = each of the {len(cat)} groups, bold = category median",
             fontsize=12)
plt.tight_layout(rect=(0, 0, 0.9, 0.96))
fig.savefig(OUT_DIR / f"scurves_{CURVE_METHOD}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# healthy vs cancer on one axis: category medians, solid = healthy, dashed = cancer
SAMPLE_STYLES = {"healthy": "-", "cancer": "--"}
fig, ax = plt.subplots(figsize=(9, 5))
ax.axvspan(*BAND, color="gold", alpha=0.3, zorder=0)
ax.axhline(0, color="k", lw=0.6)
for k in ["blood/immune", "other"] + CONTROL_CATS:
    color = "dimgray" if k == "other" else CAT_COLORS[k]
    for s in SAMPLES:
        ax.plot(periods, cat_median(curves[s], k), color=color, lw=2.2, ls=SAMPLE_STYLES[s], alpha=0.9)
handles = ([Line2D([], [], color=("dimgray" if k == "other" else CAT_COLORS[k]), lw=2.2, label=k)
            for k in ["blood/immune", "other"] + CONTROL_CATS]
           + [Line2D([], [], color="k", lw=2.2, ls=ls, label=s) for s, ls in SAMPLE_STYLES.items()])
ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_xlabel("period (bp)"); ax.set_ylabel("median correlation with expression (r)")
ax.set_title(f"Healthy vs cancer, category medians ({CURVE_METHOD})", fontsize=12)
plt.tight_layout(rect=(0, 0, 0.82, 1))
fig.savefig(OUT_DIR / f"scurves_healthy_vs_cancer_{CURVE_METHOD}.png", dpi=150, bbox_inches="tight")
plt.show()